In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

In [0]:
team_name="team_lemma"
catalog_name=f"charles_schwab_retailbrokerage_dev_{team_name}"
staging_schema=f"staging"
dbutils.widgets.text("batch_id","1","BATCH ID")
staging_prospect=f"{catalog_name}.{staging_schema}.prospect_current"
silver_prospect=f"{catalog_name}.silver.prospect"

In [0]:
batch_id=dbutils.widgets.get("batch_id")

In [0]:

df_staging = spark.read.table(staging_prospect)
df_staging=df_staging.withColumn("is_active",lit(True))\
    .withColumn("_load_ts",current_timestamp())



In [0]:
# df_staging.printSchema()

In [0]:
try:
    if spark.catalog.tableExists(silver_prospect):
        df_staging.createOrReplaceTempView("df_staging")
        print("Merging into the silver table")
        spark.sql(f"""
                MERGE INTO {silver_prospect} t 
                USING df_staging s
                ON t.AgencyID = s.AgencyID
                When matched and s.CDC_Action='C' then
                update set 
                    t.row_hash=s.row_hash,
                    t.FirstName=s.FirstName,
                    t.LastName=s.LastName,
                    t.MiddleInitial=s.MiddleInitial,
                    t.Gender=s.Gender,
                    t.Age=s.Age,
                    t.MaritalStatus=s.MaritalStatus,
                    t.AddressLine1=s.AddressLine1,
                    t.AddressLine2=s.AddressLine2,
                    t.City=s.City,
                    t.State=s.State,
                    t.PostalCode=s.PostalCode,
                    t.Country=s.Country,
                    t.Phone=s.Phone,
                    t.Income=s.Income,
                    t.NetWorth=s.NetWorth,
                    t.CreditRating=s.CreditRating,
                    t.NumberCreditCards=s.NumberCreditCards,
                    t.OwnOrRentFlag=s.OwnOrRentFlag,
                    t.NumberChildren=s.NumberChildren,
                    t.NumberCars=s.NumberCars,
                    t.Employer=s.Employer,
                    t._batch=s._batch,
                    t._source_file=s._source_file,
                    t._run_id=s._run_id,
                    t._ingest_ts=s._ingest_ts,
                    t.first_batchid=s.first_batchid,
                    t._load_ts=s._load_ts,
                    t.is_active=true
                when not matched by target and s.CDC_Action='N' then
                    insert(
                    AgencyID,row_hash,FirstName,LastName,MiddleInitial,Gender,Age,
                    MaritalStatus,AddressLine1,AddressLine2,City,State,PostalCode,
                    Country,Phone,Income,NetWorth,CreditRating,NumberCreditCards,
                    OwnOrRentFlag,NumberChildren,NumberCars,Employer,_batch,
                    _source_file,_run_id,_ingest_ts,first_batchid,_load_ts,is_active
                    )
                    values(
                    s.AgencyID,s.row_hash,s.FirstName,s.LastName,s.MiddleInitial,
                    s.Gender,s.Age,s.MaritalStatus,s.AddressLine1,s.AddressLine2,
                    s.City,s.State,s.PostalCode,s.Country,s.Phone,s.Income,
                    s.NetWorth,s.CreditRating,s.NumberCreditCards,s.OwnOrRentFlag,
                    s.NumberChildren,s.NumberCars,s.Employer,s._batch,s._source_file,
                    s._run_id,s._ingest_ts,s.first_batchid,s._load_ts,true
                    )
                when not matched by source then 
                update set 
                    t.is_active=false,
                    _load_ts=current_timestamp()

                  """)
        print("Merge completed")
        operation_type = "MERGE"

    else:
        print("table is creating")
        df_staging=df_staging.drop("CDC_Action")
        df_staging.write.format("delta").mode("overwrite").saveAsTable(silver_prospect)
        print("Table created successfully")
        operation_type = "OVERWRITE"
    staging_history = spark.sql(f"DESCRIBE HISTORY {staging_prospect}").first()
    source_count = int(staging_history["operationMetrics"].get("numOutputRows", 0))
    target_count = spark.read.table(silver_prospect).count()

    silver_history = spark.sql(f"DESCRIBE HISTORY {silver_prospect}").first()
    metrics = silver_history["operationMetrics"]
    run_id=df_staging.select("_run_id").first()[0]

    if operation_type=="MERGE":
        inserted=int(metrics.get("numTargetRowsInserted", 0))
        updated=int(metrics.get("numTargetRowsUpdated", 0))
        deleted=int(metrics.get("numTargetRowsDeleted", 0))
        rows_affected=inserted+updated+deleted
    else: 
        rows_affected = int(metrics.get("numOutputRows", 0))

    log_pipeline_recon(
        spark=spark,
        run_id=run_id,
        batch_id=batch_id,
        domain="CUSTOMER",
        table_name="prospect",
        source_layer="staging",
        target_layer="silver",
        source_count=source_count,
        target_count=target_count
    )
    
    log_audit_event(
        spark=spark,
        run_id=run_id,
        batch=batch_id, 
        layer="silver",
        table_name="prospect",
        operation=operation_type,
        rows_affected=rows_affected
    )
except Exception as e:
    print(e)


In [0]:
df_silver = spark.read.table(silver_prospect)
active_count = df_silver.filter(col("is_active") == True).count()
print(active_count)